# Latent Readout And Rollout Sweep Summary

Compact cached-results notebook for supervisor-facing plots. It loads CSV exports only and summarizes each dataset separately. The current report filters out 100-frames-per-network configurations.

Metrics shown:

1. Initial latent coordinates as a readout of final p-ratio.
2. Autoregressive rollout p-ratio R2 over 10..100 rollout steps.
3. Autoregressive rollout position MSE, capped and shown on a log scale so catastrophic failures do not hide the useful small-error range.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lss.latent.paper import (
    additive_factor_report, marginal_table, plot_factor_grid,
    select_simple_strong_rollouts, top_table,
)

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 250,
    'font.size': 9,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.24,
})

base_results = Path('../results/latent_space_capacity_sweep/full_sweep')
if not (base_results / 'exports').exists():
    base_results = Path('notebooks/results/latent_space_capacity_sweep/full_sweep')
export_dir = base_results / 'exports'

initial_run_path = export_dir / 'initial_latent_pratio_readout_by_run.csv'
rollout_run_path = export_dir / 'rollout_stats.csv'

initial_runs = pd.read_csv(initial_run_path)
rollout_runs = pd.read_csv(rollout_run_path)

excluded_train_frames_per_network = {100}
if excluded_train_frames_per_network:
    initial_before = len(initial_runs)
    rollout_before = len(rollout_runs)
    initial_runs = initial_runs[~initial_runs['train_frames_per_network'].isin(excluded_train_frames_per_network)].copy()
    rollout_runs = rollout_runs[~rollout_runs['train_frames_per_network'].isin(excluded_train_frames_per_network)].copy()
    print(
        'excluded train_frames_per_network:',
        sorted(excluded_train_frames_per_network),
        f'initial rows {initial_before} -> {len(initial_runs)};',
        f'rollout rows {rollout_before} -> {len(rollout_runs)}',
    )

factors = ['latent_dim', 'train_networks', 'train_frames_per_network']
plot_specs = [
    ('latent_dim', 'CV count'),
    ('train_networks', 'Training networks'),
    ('train_frames_per_network', 'Frames / network'),
]
dataset_order = [d for d in ['reid', 'depablo'] if d in set(rollout_runs['dataset_name'])]
dataset_labels = {'reid': 'Reid', 'depablo': 'de Pablo'}

mse_plot_floor = 1e-8
mse_plot_cap = 1e-4

print('exports:', export_dir)
print('initial readout rows:', initial_runs.shape)
print('rollout rows:', rollout_runs.shape)
print('datasets:', dataset_order)



In [ ]:
# Aggregate initial latent -> final p-ratio readout by config and dataset.
# combo_test_r2 is the test R2 after fitting a linear CV combination on validation.
initial_summary = (
    initial_runs
    .groupby(['dataset_name', 'target_mode'] + factors, as_index=False)
    .agg(
        combo_test_r2_mean=('combo_test_r2', 'mean'),
        combo_test_r2_std=('combo_test_r2', 'std'),
        single_test_r2_mean=('single_test_r2', 'mean'),
        single_test_r2_std=('single_test_r2', 'std'),
        n_repeats=('repeat_idx', 'nunique'),
    )
)

# Aggregate rollout metrics from per-repeat rows, then summarize each config over steps 10..100.
rollout_runs = rollout_runs.copy()
rollout_runs['rollout_p_ratio_r2_clipped'] = rollout_runs['p_ratio_r2'].clip(lower=0)
rollout_runs['position_mse_plot'] = rollout_runs['final_pos_mse'].clip(lower=mse_plot_floor, upper=mse_plot_cap)

rollout_step_summary = (
    rollout_runs
    .groupby(['dataset_name', 'target_mode'] + factors + ['rollout_steps'], as_index=False)
    .agg(
        rollout_p_ratio_r2_mean=('p_ratio_r2', 'mean'),
        rollout_p_ratio_r2_std=('p_ratio_r2', 'std'),
        rollout_p_ratio_r2_clipped_mean=('rollout_p_ratio_r2_clipped', 'mean'),
        rollout_position_mse_mean=('final_pos_mse', 'mean'),
        rollout_position_mse_plot_mean=('position_mse_plot', 'mean'),
        n_repeats=('repeat_idx', 'nunique'),
        used=('used', 'mean'),
    )
)

rollout_config_summary = (
    rollout_step_summary
    .groupby(['dataset_name', 'target_mode'] + factors, as_index=False)
    .agg(
        mean_rollout_r2=('rollout_p_ratio_r2_mean', 'mean'),
        mean_rollout_r2_clipped=('rollout_p_ratio_r2_clipped_mean', 'mean'),
        min_rollout_r2=('rollout_p_ratio_r2_mean', 'min'),
        mean_position_mse=('rollout_position_mse_mean', 'mean'),
        mean_position_mse_plot=('rollout_position_mse_plot_mean', 'mean'),
        n_repeats=('n_repeats', 'min'),
    )
)

step50 = rollout_step_summary[rollout_step_summary['rollout_steps'].eq(50)][
    ['dataset_name', 'target_mode'] + factors + ['rollout_p_ratio_r2_mean', 'rollout_position_mse_mean', 'rollout_position_mse_plot_mean']
].rename(columns={
    'rollout_p_ratio_r2_mean': 'step50_rollout_r2',
    'rollout_position_mse_mean': 'step50_position_mse',
    'rollout_position_mse_plot_mean': 'step50_position_mse_plot',
})
step100 = rollout_step_summary[rollout_step_summary['rollout_steps'].eq(100)][
    ['dataset_name', 'target_mode'] + factors + ['rollout_p_ratio_r2_mean', 'rollout_position_mse_mean', 'rollout_position_mse_plot_mean']
].rename(columns={
    'rollout_p_ratio_r2_mean': 'step100_rollout_r2',
    'rollout_position_mse_mean': 'step100_position_mse',
    'rollout_position_mse_plot_mean': 'step100_position_mse_plot',
})
rollout_config_summary = rollout_config_summary.merge(step50, on=['dataset_name', 'target_mode'] + factors, how='left').merge(
    step100, on=['dataset_name', 'target_mode'] + factors, how='left'
)

print('Initial readout repeat coverage')
initial_coverage = initial_summary.groupby('dataset_name')['n_repeats'].agg(['count', 'min', 'max'])
display(initial_coverage)
if (initial_coverage['min'] < 3).any():
    print('Note: initial latent readout uses whatever cached scan exists. Rerun/refresh the initial readout scan if you need 3-repeat initial readout for every dataset.')
print('Rollout repeat coverage')
display(rollout_config_summary.groupby('dataset_name')['n_repeats'].agg(['count', 'min', 'max']))


In [ ]:
# Quantify each sweep factor with a shared additive analysis.
factor_reports = {
    'initial_combo_r2': additive_factor_report(initial_summary, 'combo_test_r2_mean', factors),
    'rollout_pratio_r2': additive_factor_report(rollout_config_summary, 'mean_rollout_r2_clipped', factors),
    'rollout_position_mse_plot': additive_factor_report(rollout_config_summary, 'mean_position_mse_plot', factors),
}

for name, report in factor_reports.items():
    print(name)
    display(report.round(4))


In [ ]:
# Shared factor plots.
plot_factor_grid(
    initial_summary, 'combo_test_r2_mean',
    dataset_order=dataset_order, dataset_labels=dataset_labels, plot_specs=plot_specs,
    title='Initial latent readout of final p-ratio',
    ylabel='test R2',
    color='#2364aa',
    ylim=(0, 1.0),
)

plot_factor_grid(
    rollout_config_summary, 'mean_rollout_r2_clipped',
    dataset_order=dataset_order, dataset_labels=dataset_labels, plot_specs=plot_specs,
    title='Autoregressive rollout p-ratio R2, mean over 10..100 steps',
    ylabel='R2, negative clipped to 0',
    color='#d95f02',
    ylim=(0, 1.0),
)

plot_factor_grid(
    rollout_config_summary, 'mean_position_mse_plot',
    dataset_order=dataset_order, dataset_labels=dataset_labels, plot_specs=plot_specs,
    title=f'Autoregressive rollout position MSE, clipped to [{mse_plot_floor:g}, {mse_plot_cap:g}]',
    ylabel='position MSE, log scale',
    color='#2a9d8f',
    ylim=(mse_plot_floor, mse_plot_cap),
    log_y=True,
    clip_errors=True, value_floor=mse_plot_floor, value_cap=mse_plot_cap,
)


In [ ]:
# Top configurations by dataset.
for dataset_name in dataset_order:
    label = dataset_labels.get(dataset_name, dataset_name)
    print(f'===== {label}: top 5 initial latent p-ratio readout =====')
    display(top_table(
        initial_summary,
        dataset_name,
        'combo_test_r2_mean',
        False,
        factors + ['combo_test_r2_mean', 'combo_test_r2_std', 'single_test_r2_mean', 'n_repeats'],
    ))

    print(f'===== {label}: top 5 rollout p-ratio R2, mean over 10..100 =====')
    display(top_table(
        rollout_config_summary,
        dataset_name,
        'mean_rollout_r2',
        False,
        factors + ['mean_rollout_r2', 'min_rollout_r2', 'step50_rollout_r2', 'step100_rollout_r2', 'mean_position_mse', 'n_repeats'],
    ))

    print(f'===== {label}: top 5 rollout position MSE, mean over 10..100; lower is better =====')
    display(top_table(
        rollout_config_summary,
        dataset_name,
        'mean_position_mse',
        True,
        factors + ['mean_position_mse', 'step50_position_mse', 'step100_position_mse', 'mean_rollout_r2', 'step100_rollout_r2', 'n_repeats'],
    ))

    print(f'===== {label}: top 5 rollout p-ratio R2 at exactly 50 steps =====')
    display(top_table(
        rollout_config_summary,
        dataset_name,
        'step50_rollout_r2',
        False,
        factors + ['step50_rollout_r2', 'step50_position_mse', 'mean_rollout_r2', 'mean_position_mse', 'n_repeats'],
    ))


## Best Rollout Curves

For each dataset, select five strong autoregressive rollout configurations with a simplicity preference. The selection first keeps configurations within `0.05` mean p-ratio R2 of the best configuration for that dataset, then prefers lower CV dimension, fewer training networks, and fewer frames per network. The plots show p-ratio R2 as a function of rollout step; tables list the corresponding summary metrics.


In [ ]:
# Best rollout candidates: strong performance with a simplicity preference.
selection_tolerance = 0.05
n_selected_rollouts = 5

selected_rollout_parts = []
for dataset_name in dataset_order:
    selected = select_simple_strong_rollouts(
        rollout_config_summary,
        dataset_name,
        factors=factors,
        n=n_selected_rollouts,
        tolerance=selection_tolerance,
    )
    selected_rollout_parts.append(selected)
    label = dataset_labels.get(dataset_name, dataset_name)
    print(f'===== {label}: selected simple strong rollout configs =====')
    display(selected[
        factors + [
            'mean_rollout_r2',
            'min_rollout_r2',
            'step50_rollout_r2',
            'step100_rollout_r2',
            'mean_position_mse',
            'step50_position_mse',
            'step100_position_mse',
            'delta_from_best',
            'n_repeats',
        ]
    ].round(6))

selected_rollouts = pd.concat(selected_rollout_parts, ignore_index=True) if selected_rollout_parts else pd.DataFrame()

fig, axes = plt.subplots(1, len(dataset_order), figsize=(5.8 * len(dataset_order), 4.2), sharey=True, squeeze=False, constrained_layout=True)
for ax, dataset_name in zip(axes[0], dataset_order):
    selected = selected_rollouts[selected_rollouts['dataset_name'].eq(dataset_name)].copy()
    ds_steps = rollout_step_summary[rollout_step_summary['dataset_name'].eq(dataset_name)].copy()
    for _, row in selected.iterrows():
        mask = np.ones(len(ds_steps), dtype=bool)
        for factor in factors:
            mask &= ds_steps[factor].eq(row[factor]).to_numpy()
        curve = ds_steps[mask].sort_values('rollout_steps')
        if curve.empty:
            continue
        label = f"{row['config_label']} | mean={row['mean_rollout_r2']:.2f}"
        ax.plot(curve['rollout_steps'], curve['rollout_p_ratio_r2_mean'], marker='o', lw=1.8, ms=4, label=label)
        y = curve['rollout_p_ratio_r2_mean'].to_numpy(dtype=float)
        yerr = curve['rollout_p_ratio_r2_std'].fillna(0).to_numpy(dtype=float)
        ax.fill_between(curve['rollout_steps'], y - yerr, y + yerr, alpha=0.10)
    ax.axhline(0.0, color='0.25', lw=0.9, alpha=0.7)
    ax.set_ylim(-0.05, 1.02)
    ax.set_xlabel('rollout steps')
    ax.set_title(dataset_labels.get(dataset_name, dataset_name))
    ax.grid(alpha=0.24)
    ax.legend(frameon=False, fontsize=7, loc='lower right')
axes[0, 0].set_ylabel('p-ratio R2 on test rollouts')
fig.suptitle('Best simple autoregressive rollout candidates: p-ratio R2 vs rollout step', fontsize=12, fontweight='bold')
plt.show()

# Same selected configs, position MSE versus rollout step on a clipped log scale.
fig, axes = plt.subplots(1, len(dataset_order), figsize=(5.8 * len(dataset_order), 4.2), sharey=True, squeeze=False, constrained_layout=True)
for ax, dataset_name in zip(axes[0], dataset_order):
    selected = selected_rollouts[selected_rollouts['dataset_name'].eq(dataset_name)].copy()
    ds_steps = rollout_step_summary[rollout_step_summary['dataset_name'].eq(dataset_name)].copy()
    for _, row in selected.iterrows():
        mask = np.ones(len(ds_steps), dtype=bool)
        for factor in factors:
            mask &= ds_steps[factor].eq(row[factor]).to_numpy()
        curve = ds_steps[mask].sort_values('rollout_steps')
        if curve.empty:
            continue
        mse = curve['rollout_position_mse_plot_mean'].clip(lower=mse_plot_floor, upper=mse_plot_cap)
        ax.plot(curve['rollout_steps'], mse, marker='o', lw=1.8, ms=4, label=row['config_label'])
    ax.set_yscale('log')
    ax.set_ylim(mse_plot_floor, mse_plot_cap)
    ax.axhline(mse_plot_cap, color='0.35', lw=1, ls=':', alpha=0.75)
    ax.set_xlabel('rollout steps')
    ax.set_title(dataset_labels.get(dataset_name, dataset_name))
    ax.grid(alpha=0.24)
    ax.legend(frameon=False, fontsize=7, loc='best')
axes[0, 0].set_ylabel('position MSE, clipped log scale')
fig.suptitle('Selected rollout candidates: position MSE vs rollout step', fontsize=12, fontweight='bold')
plt.show()

# Reference: pure top-5 by mean rollout R2, without simplicity preference.
for dataset_name in dataset_order:
    label = dataset_labels.get(dataset_name, dataset_name)
    print(f'===== {label}: pure top 5 by mean rollout R2 =====')
    display(
        rollout_config_summary[rollout_config_summary['dataset_name'].eq(dataset_name)]
        .sort_values('mean_rollout_r2', ascending=False)
        .head(5)[factors + ['mean_rollout_r2', 'min_rollout_r2', 'step50_rollout_r2', 'step100_rollout_r2', 'mean_position_mse', 'n_repeats']]
        .round(6)
    )


In [ ]:
# Compact numeric marginals for paper notes.
print('Initial readout R2 marginals')
display(marginal_table(initial_summary, 'combo_test_r2_mean', dataset_order=dataset_order, factors=factors).round(4))
print('Rollout p-ratio R2 marginals')
display(marginal_table(rollout_config_summary, 'mean_rollout_r2_clipped', dataset_order=dataset_order, factors=factors).round(4))
print('Rollout position MSE marginals, clipped for plotting')
display(marginal_table(rollout_config_summary, 'mean_position_mse_plot', dataset_order=dataset_order, factors=factors).round(8))
